# Лабораторная работа 6. Создание чатбота на основе LLM

#### Работу выполнил: Самойло Александр, студент группы ФИб-4

**Для повышения скорости работы подключите GPU в меню `Среда выполнения`. LLM и её входные данные загружайте на видеокарту.**

# Чать 1. Знакомство с библиотекой `transformers`

Классы `AutoTokenizer` и `AutoModelForCausalLM` позволяет загрузить чекпоинты языковой модели и выполнить генерацию текста.

In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import gradio

В качестве модели для экспериментов возьмите модель `Qwen/Qwen3-0.6B` с huggingface. Познакомьтесь с описанием модели и её использованием [ссылка](https://huggingface.co/Qwen/Qwen3-0.6B). Веса загружайте в типе `torch.bfloat16` для экономии памяти GPU. Не забудьте в интерфейсе colab подключить GPU.

In [18]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name_or_path = "Qwen/Qwen3-0.6B"
# Убираем device_map="cuda", переносим модель на девайс вручную после загрузки
model = AutoModelForCausalLM.from_pretrained(model_name_or_path, torch_dtype=torch.bfloat16)
model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Входной текст для генерации может быть представлен в режиме диалога в виде списка объектов с полями `role` и `content`. Значения поля `role` может принимать значения `system`, `user`, `assistent`, что соответсвует системному, пользовательскому промптам и ответу модели. У каждой модели специальные токены и собственный формат, приведение к которому происходит с помощью метода `apply_chat_template()`

In [19]:
chat = [
  {"role": "system", "content": "Отвечай на русском языке"},
  {"role": "user", "content": "Что ты думаешь о законах робототехники?"},
  {"role": "assistant", "content": "Законы робототехники — это нарастающая область правовой регуляции, которая стремится адаптироваться к быстрому развитию технологий в области искусственного интеллекта (ИИ) и автоматизации."},
  {"role": "user", "content": "А какой второй закон?"}
]
text = tokenizer.apply_chat_template(chat, tokenize=False,
                                       add_generation_prompt=True,
                                       return_tensors="pt"
)
print(text)

<|im_start|>system
Отвечай на русском языке<|im_end|>
<|im_start|>user
Что ты думаешь о законах робототехники?<|im_end|>
<|im_start|>assistant
Законы робототехники — это нарастающая область правовой регуляции, которая стремится адаптироваться к быстрому развитию технологий в области искусственного интеллекта (ИИ) и автоматизации.<|im_end|>
<|im_start|>user
А какой второй закон?<|im_end|>
<|im_start|>assistant



Генерация выполняется с помощью метода generate(). Генерируемый текст большой (с рассуждениями), поэтому раскройте поле вывода.

In [20]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)

In [21]:
all_generated_text = tokenizer.decode(generated_ids[0])
print(all_generated_text)

<|im_start|>system
Отвечай на русском языке<|im_end|>
<|im_start|>user
Что ты думаешь о законах робототехники?<|im_end|>
<|im_start|>assistant
Законы робототехники — это нарастающая область правовой регуляции, которая стремится адаптироваться к быстрому развитию технологий в области искусственного интеллекта (ИИ) и автоматизации.<|im_end|>
<|im_start|>user
А какой второй закон?<|im_end|>
<|im_start|>assistant
<think>
Хорошо, пользователь спросил, какой второй закон я думаю о законах робототехники. Поскольку в первом ответе я упомянул "законы робототехники", то второй закон должен быть более конкретным или разным. Возможно, пользователь хочет узнать о других аспектах, связанных с робототехникой.

Нужно проверить, что пользователь действительно спрашивает о втором законе. Если в первом ответе я описал общие законы, то второй закон может быть о правовых нормах или конкретных правилах. Например, законы о безопасности, лицензировании или применении прав в автоматизации.

Стоит также уточнит

Изучите формат ответа модели и распарсите ответ, например следующим образом.

In [6]:
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
print(f"Рассуждения: {thinking_content}")
print(f"Ответ: {content}")

Рассуждения: <think>
Хорошо, пользователь спрашивает о втором законах робототехники. Нужно уточнить, что он ожидает от меня, иначе ответ может быть не соответствующим. Сначала проверю, что я правильно понял вопрос. В предыдущем ответе я говорил о законах робототехники как области правовой регуляции, адаптирующейся к развитию ИИ. Теперь пользователь спрашивает о втором законах, возможно, имея в виду конкретный закон, который я ранее описал.

Надо понять, что он хочет узнать о конкретном законе, но я не могу предоставить конкретные законодательные акты без дополнительной информации. Возможно, он подразумевает, что законы робототехники связаны с конкретными нормами, например, о безопасности, этике или инновациях. В этом случае нужно уточнить, какие именно законы он ожидает. Также стоит упомянуть, что законы робототехники постоянно меняются, и в зависимости от контекста могут быть разные точки зрения.

Также важно сохранить ответ в рамках русского языка и обеспечить полную информацию, даже

**Задание 1.** Напишите чатбот c поддержкой истории при генерации ответа. Для ввода сообщений пользователя используйте функцию `input()`, в случае введения пользователем текста `выход`, диалог должен останавливаться, а исполнение кода прекращаться.

Глубина истории должна задаваться параметром.

Сделайте логирование входных данных модели в текстовый файл и проверьте правильность отправляемых данных в модель для генерации.

Поддержку истории проверяйте следующим диалогом при установленной длине истории 1 (одно предыдущее сообщение) или при большей длине, но тогда адаптируйте диалог:
```
USER: Какой первый закон робототехники?
ASSISTANT: ....
USER: А какой второй закон?
ASSISTANT:
```

In [23]:
import json

history_depth = 2 # Количество запоминаемых предыдущих пар вопрос-ответ
chat_history =[]
system_prompt = {"role": "system", "content": "Отвечай на русском языке кратко и понятно."}

while True:
    user_text = input("USER: ")

    if user_text.strip().lower() == "выход":
        print("Чат завершен.")
        break

    chat_history.append({"role": "user", "content": user_text})

    # Ограничиваем длину истори
    messages_to_keep = history_depth * 2 + 1

    if len(chat_history) > messages_to_keep:
        chat_history = chat_history[-messages_to_keep:]

    messages_to_send = [system_prompt] + chat_history

    with open("chat_log.txt", "a", encoding="utf-8") as f:
        f.write("Отправлено в модель: " + json.dumps(messages_to_send, ensure_ascii=False) + "\n")

    # Подготовка данных для модели
    text = tokenizer.apply_chat_template(messages_to_send, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Генерация ответа
    generated_ids = model.generate(**model_inputs, max_new_tokens=1024)
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    try:
        index = len(output_ids) - output_ids[::-1].index(151668) # поиск </think>
    except ValueError:
        index = 0

    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    print(f"ASSISTANT: {content}")

    chat_history.append({"role": "assistant", "content": content})

USER: какой первый закон термодинамики
ASSISTANT: Первый закон термодинамики: теплота и работа связаны, так как теплота может передаваться, а работа — изменять состояние системы.
USER: а второй
ASSISTANT: Второй закон термодинамики: энергия не может быть преобразована обратно в тепло при выполнении работы.
USER: выход
Чат завершен.


# Часть 2. Знакомство с `langchain`. Простой пример RAG

В colab-сессии скорее всего уже установлены требуемые модули, но если нет, то скорее всего потребуются следующие.

In [2]:
!pip install -U langchain langchain-core
!pip install -U langchain-community langchain-text-splitters
!pip install -U langchain-huggingface sentence-transformers

Выполним импорт требуемых модулей.

In [3]:
from langchain_community.vectorstores import Chroma #векторная бд
from langchain_text_splitters import RecursiveCharacterTextSplitter#нарезчик текста
from langchain_huggingface import HuggingFaceEmbeddings#переводчик смысла в цифры
from langchain_classic.chains import RetrievalQA#связующее звено
from langchain_classic.document_loaders import DirectoryLoader, JSONLoader #грузчики

Также установим дополнительные модули.

In [4]:
!pip install jq
!pip install chromadb

Определим переменные для каталога с документами, по которым будет выполняться поиск, и для каталога под векторную базу данных.

In [5]:
data_directory = "/content/documents"
db_directory = "/content/chroma_db"
load_new_db = False

Загрузим `csv`-файл новостного датасета, который использовали ранее заданиях, и пересохраним данные в `json`-файл.

In [6]:
import pandas as pd
df = pd.read_csv('news_lemmatized.csv', sep=',')
df.to_json('lenta_ru_news_filtered.json', orient='records', force_ascii=False, indent=4)

Переместите полученный `json`-файл с текстами в `data_directory`.

Инициализируем модель, с помощью которой будут строиться эмбеддинги текстовых фагментов документов.

In [7]:
data_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Загрузим документы. Пусть документы представляют собой `json` файлы со списками объектов полем `text`, в котором хранится текст (пример файла находится в гугл-папке вместе с блокнотом лабораторной работы).

In [8]:
import os
import shutil
os.makedirs('/content/documents/', exist_ok=True)
shutil.move('lenta_ru_news_filtered.json', '/content/documents/lenta_ru_news_filtered.json')

'/content/documents/lenta_ru_news_filtered.json'

In [9]:
loader = DirectoryLoader(data_directory, glob="**/*.json", show_progress=True, loader_cls=JSONLoader, loader_kwargs={"jq_schema":".[].text"})
#jq_schema - инструкция для извлечения текста: берёт все содержимое в поле text из каждого json массива
documents = loader.load()
print(f"Count of documents is {len(documents)}")

100%|██████████| 1/1 [00:02<00:00,  2.08s/it]

Count of documents is 4000


Разделим тексты на фрагменты.

In [10]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=10)
#режем на куски по 200 символов, нахлест 10 (повторяет 10 символов последнего)
texts = text_splitter.split_documents(documents) #кол-во документов стало больше
print(f"Count of hunks is {len(texts)}")

Count of hunks is 28876


Наполним векторную базу данных.

In [11]:
vector_db = Chroma.from_documents(documents=texts, embedding=data_embeddings, persist_directory=db_directory)#посл: сохраняем базу на диск чтобы не пересчитывать

Создадим объект, через который будет выполняться поиск.\
`search_type="mmr"` - это метод MMR (**Maximal Marginal Relevance** — максимальная маржинальная релевантность). В отличие от стандартного поиска по сходству, который просто выдает самые похожие документы, MMR старается найти баланс между релевантностью запросу и разнообразием ответов.

In [12]:
retriever = vector_db.as_retriever(search_type="mmr")#возвращает разнообразные наборы релев. данных

Зададим запрос и выполним поиск.

In [13]:
query = "Самая высокооплачиваемая писательница"
response = retriever.invoke(query)#превращение запроса в вектор
results = [{"id":i, "text":x.page_content, "source":x.metadata['source']} for i,x in enumerate(response)]
for x in results:
  print(x)

{'id': 0, 'text': 'Американский журнал Forbes составил ежегодный список самых высокооплачиваемых писателей в США. Лидером рейтинга стал Джеймс Паттерсон, автор детективов об инспекторе Алексе Кроссе. Писатель заработал', 'source': '/content/documents/lenta_ru_news_filtered.json'}
{'id': 1, 'text': 'сейчас идет подготовка к похоронам. Харлан Эллисон считался одним из самых плодовитых писателей. Среди его работ получившие различные награды рассказы «У меня нет рта, но я должен кричать», «Джеффти', 'source': '/content/documents/lenta_ru_news_filtered.json'}
{'id': 2, 'text': 'из самых высокооплачиваемых диджеев в мире.', 'source': '/content/documents/lenta_ru_news_filtered.json'}
{'id': 3, 'text': 'назвал информацию о своем доходе в 174 миллиона рублей «полным бредом». «Мой оклад как руководителя галереи — 80 тысяч рублей. Плюс премии. Иногда я пишу картины на заказ. Но чтобы заработать 174', 'source': '/content/documents/lenta_ru_news_filtered.json'}


**Задание 2.** Реализуйте поиск по новостным текстам из датасета предыдущих лабораторных работ. Протестируйте работу поиска. Сравните реализованный поиск с поиском на основе `TF-IDF` из предыдущих лабораторных работ, используя прежние запросы. Сделайте выводы.

In [14]:
test_queries =[
    "Самые богатые создатели детективных романов",
    "Сколько зарабатывают художники и директора выставок",
    "Создатели электронной танцевальной музыки"
]

queries= ["банк инвестиции фондовый рынок",
        "кино актер режиссер премия",
        "финансирование театров и выставок",
        "футбол гол тренер матч",
        "Самый высокооплачиваемый писатель"]

for query in queries:
    print(f"Поиск по запросу: «{query}»")

    # invoke() возвращает список найденных чанков
    docs = retriever.invoke(query)

    #топ-3 результат
    for i, doc in enumerate(docs[:3]):
        clean_text = doc.page_content.replace('\n', ' ')
        print(f"  [{i+1}] Результат: {clean_text[:150]}...")

    print("-" * 50)

Поиск по запросу: «банк инвестиции фондовый рынок»
  [1] Результат: рынках инвестиционных паев ЗПИФ акций «ФИНАМ — Информационные технологии» и ЗПИФ рентный «Капитальные вложения» значительная доля торгов пришлась на с...
  [2] Результат: активы Сбербанка....
  [3] Результат: периодом прошлого года. При этом госбанк рассчитывает на дальнейшее увеличение портфеля. В рамках акции минимальная ставка составит 11,7 процента, а с...
--------------------------------------------------
Поиск по запросу: «кино актер режиссер премия»
  [1] Результат: в истории отечественного кино....
  [2] Результат: яиншлагбаума#городволгоград#волгоград#...
  [3] Результат: наук, президента холдинга «Ромир» Андрея Милехина....
--------------------------------------------------
Поиск по запросу: «финансирование театров и выставок»
  [1] Результат: артистов. Часть средств от продажи билетов на концерт Анны Нетребко и других международных оперных звезд передадут в фонд. В частности, благотворитель...
  [2] Результа

In [ ]:
'''
Сравнительный анализ:
Параметр сравнения	TF-IDF (Лексический поиск)	                        RAG / Векторный поиск (Семантический)
Принцип работы	    Поиск по точному совпадению слов.	                  Поиск по смысловой близости векторов.
Результаты	        Часто находит шум, если слова совпадают случайно. 	Находит документы по теме, даже с другими словами.
Гибкость	          Низкая (нужны точные формулировки).	                Высокая (понимает синонимы и контекст).
'''

# Часть 3. Чатбот с использованием внешних знаний

**Задание 3.** Напишите чатбот, в котором перед генерацией выполнятся поиск внешних знаний, а затем выполняется генерация ответа пользователю с учётом этих знаний и истории диалога (тестируйте на глубине истории не более 5 сообщений диалога). Придумайте что передавать в качестве запроса в векторную БД (например, можно попросить LLM суммаризовать историю диалога или самой сформулировать запрос из 5-10 слов) - попробуйте несколько вариантов и выберите более эффективный.

Протестируйте работу получившегося чатбота. Сделайте выводы.

In [22]:
MAX_HISTORY = 3

chat_history =[
    {"role": "system", "content": "Ты вежливый помощник. Отвечай на вопросы пользователя, используя предоставленный контекст из базы знаний."}
]

def get_llm_response(messages_list):
    text = tokenizer.apply_chat_template(messages_list, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=512, pad_token_id=tokenizer.eos_token_id)
    output_ids = outputs[0][len(inputs.input_ids[0]):].tolist()

    response = tokenizer.decode(output_ids, skip_special_tokens=True)

    # рассуждения <think> отрезаем для чистоты ответа
    if "</think>" in response:
        response = response.split("</think>")[-1].strip()
    return response


print("Введите ваш запрос ('выход' для завершения)\n")

while True:
    user_message = input("USER: ")
    if user_message.lower() == "выход":
        print("Завершение работы.")
        break

    query_prompt =[
        {"role": "system", "content": "Твоя задача — формировать короткие поисковые запросы (5-10 слов) для базы данных. Посмотри на историю диалога и последнее сообщение. Если в сообщении есть местоимения (он, она, это), замени их на суть из истории. Напиши ТОЛЬКО сам поисковый запрос, без лишних слов."},
    ]
    # добавление последних сообщений из истории для контекста
    query_prompt.extend(chat_history[1:])
    query_prompt.append({"role": "user", "content": f"Сформируй поисковый запрос для этого сообщения: {user_message}"})

    search_query = get_llm_response(query_prompt)
    print(f"  [DEBUG] Сгенерированный поисковый запрос для БД: {search_query}")
    #поиск в БД
    found_docs = retriever.invoke(search_query)
    context_text = "\n".join([doc.page_content for doc in found_docs[:3]]) # Берем топ-3 факта

    # формирование промпта, подмешивая туда найденные факты
    enriched_user_message = f"Информация из базы данных:\n{context_text}\n\nВопрос пользователя: {user_message}"

    # добавление сообщения в историю для генерации
    messages_to_model = chat_history.copy()
    messages_to_model.append({"role": "user", "content": enriched_user_message})

    # получение финального ответа
    final_answer = get_llm_response(messages_to_model)
    print(f"ASSISTANT: {final_answer}\n")

    # сохранение только оригинального вопроса юзера (без служебного контекста БД, чтобы не засорять память)
    chat_history.append({"role": "user", "content": user_message})
    chat_history.append({"role": "assistant", "content": final_answer})

    # обрез истории, если она превысила MAX_HISTORY (оставляем System + последние N пар)
    if len(chat_history) > (MAX_HISTORY * 2) + 1:
         chat_history = [chat_history[0]] + chat_history[-(MAX_HISTORY * 2):]

Введите ваш запрос ('выход' для завершения)

USER: самый высокооплачиваемый писатель
  [DEBUG] Сгенерированный поисковый запрос для БД: самый высокооплачиваемый писатель
ASSISTANT: Лидер Forbes-еconomический рейтинг стал Джеймс Паттерсон, автор детективов, который заработал более 58,4 миллиона в 2013 году.

USER: выход
Завершение работы.
